# Metadata Column Ingestion from Cloud Files

## Overview
When ingesting data from cloud storage, Databricks automatically provides metadata columns that contain information about the source file.

## Available Metadata Columns
- `_metadata.file_path`: Full path to the source file
- `_metadata.file_name`: Name of the source file
- `_metadata.file_size`: Size of the file in bytes
- `_metadata.file_modification_time`: Last modification timestamp

## Why Metadata Columns Matter
- **Data Lineage**: Track which file each record came from
- **Incremental Loads**: Identify new files for processing
- **Data Quality**: Monitor file characteristics
- **Compliance**: Audit trail of data sources

## Step 1: Import Libraries

In [ ]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

## Step 2: Define Schema for Data

In [ ]:
schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("payment_method", StringType(), True),
    StructField("status", StringType(), True)
])
print("Schema defined successfully")

## Step 3: Create Table with Metadata Columns

In [ ]:
DROP TABLE IF EXISTS transactions_with_metadata;

CREATE TABLE transactions_with_metadata (
  transaction_id STRING,
  customer_id STRING,
  amount DOUBLE,
  transaction_date DATE,
  payment_method STRING,
  status STRING,
  ingestion_timestamp TIMESTAMP,
  source_file_path STRING,
  source_file_name STRING,
  source_file_size LONG,
  source_file_mod_time TIMESTAMP
)
USING DELTA

## Step 4: COPY INTO with Metadata Columns

In [ ]:
COPY INTO transactions_with_metadata
FROM (
  SELECT 
    transaction_id,
    customer_id,
    amount,
    CAST(transaction_date AS DATE) as transaction_date,
    payment_method,
    status,
    CURRENT_TIMESTAMP() as ingestion_timestamp,
    _metadata.file_path as source_file_path,
    _metadata.file_name as source_file_name,
    _metadata.file_size as source_file_size,
    _metadata.file_modification_time as source_file_mod_time
  FROM read_files(
    '/Volumes/data_analysis/sample_data/',
    format => 'csv'
  )
)
FILEFORMAT = CSV

## Step 5: Query with Metadata

In [ ]:
SELECT 
  transaction_id,
  customer_id,
  amount,
  source_file_name,
  source_file_size
FROM transactions_with_metadata
LIMIT 10

## Step 6: Analyze Data by Source File

In [ ]:
SELECT 
  source_file_name,
  COUNT(*) as record_count,
  source_file_size
FROM transactions_with_metadata
GROUP BY source_file_name, source_file_size
ORDER BY source_file_name

## Step 7: Auto Loader with Metadata

In [ ]:
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/tmp/autoloader_checkpoint/meta/") \
    .schema(schema) \
    .option("header", "true") \
    .load("/Volumes/data_analysis/sample_data/")

df_with_meta = df_stream \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file_path", col("_metadata.file_path")) \
    .withColumn("source_file_name", col("_metadata.file_name"))

print("Stream with metadata configured")

## Step 8: Write Stream with Metadata

In [ ]:
streaming_query = df_with_meta.writeStream \
    .format("delta") \
    .mode("append") \
    .option("checkpointLocation", "/tmp/autoloader_checkpoint/meta/") \
    .table("transactions_stream_meta")

print(f"Query started: {streaming_query.id}")

## Use Cases

### 1. Data Lineage Tracking

In [ ]:
SELECT 
  source_file_path,
  COUNT(*) as records_from_file
FROM transactions_with_metadata
GROUP BY source_file_path

### 2. Data Quality Monitoring

In [ ]:
SELECT 
  source_file_name,
  COUNT(*) as record_count,
  source_file_size,
  ROUND(source_file_size / COUNT(*), 2) as avg_bytes_per_record
FROM transactions_with_metadata
GROUP BY source_file_name, source_file_size

## Best Practices

1. Always capture metadata in production pipelines
2. Store metadata columns in Delta tables for historical tracking
3. Use metadata for incremental load identification
4. Monitor file patterns for anomalies
5. Extract dates from file names for partitioning

## Summary

Metadata columns provide:
- Complete data lineage tracking
- Efficient incremental patterns
- Data quality insights
- Compliance support

Always leverage `_metadata` columns for production-grade pipelines.